# NPKmath step 2: does verified context help an AI?
Run top to bottom. Part A checks the harness offline. Part B measures a **real** model.

Fair test rules: use the same model for both conditions, start a **new chat** for every batch, and never tell it the answers.

In [ ]:
import os, glob
if not os.path.exists('npkmath'):
    z = sorted(glob.glob('/content/npkmath_step*.zip'))[-1]
    folder = os.path.basename(z)[:-4]
    !unzip -oq {z} -d /content
    %cd /content/{folder}
!pip -q install -e ".[dev]"
!python -m pytest -q

## Part A. Build the question set (offline)

In [ ]:
import json
from npkmath.dataset import build_dataset, majority_baseline
from npkmath.engine import NPKCoreEngine
from npkmath.llm_eval import *
from npkmath.signals import AIFeedbackInterface

engine = NPKCoreEngine()
items = build_dataset(seed=0, max_items=60)
cb = ContextBuilder(items, engine)          # verifies every claim once, builds related context
print(len(items), 'questions; always-guess-the-commonest floor =', majority_baseline(items))

In [ ]:
# Harness self-test with a FAKE model. This proves the plumbing works. It is NOT evidence about any AI.
mock = MockModel(items, accuracy=0.6, seed=1, grounded_boost=0.25)
rows = {
  'baseline': score(items, collect_answers(items, mock), 'baseline'),
  'grounded': score(items, collect_answers(items, mock, cb.contexts()), 'grounded'),
}
print(json.dumps(summarize(items, rows), indent=1))

## Part B. Measure a real model (copy-paste mode, works with any chat AI)
1. Run the next cell. Copy the printed prompt into a **new** chat with your AI.
2. Paste the AI's reply into `baseline_replies` (below) under the right batch number.
3. Repeat for batches 1 and 2, then do the same with `grounded_batches`.

The AI should answer with lines like `Q01: TRUE`.

In [ ]:
base_batches = export_batches(items, None, 20)
grounded_batches = export_batches(items, cb.contexts(), 20)
print(len(base_batches), 'batches per condition')

WHICH, N = base_batches, 0      # switch to grounded_batches for condition 2; N = 0, 1, 2
print(WHICH[N][0])

In [ ]:
# Paste replies between the triple quotes. Leave '' for batches you haven't done.
baseline_replies = {
 0: '''''',
 1: '''''',
 2: '''''',
}
grounded_replies = {
 0: '''''',
 1: '''''',
 2: '''''',
}

In [ ]:
def gather(batches, replies):
    answers = {}
    for k, (prompt, qmap) in enumerate(batches):
        answers.update(parse_batch(replies.get(k, ''), qmap))
    return answers

MODEL_NAME = 'my-model'          # name the AI you tested
fb = AIFeedbackInterface()
rows = {}
for cond, batches, replies in [('baseline', base_batches, baseline_replies),
                               ('grounded', grounded_batches, grounded_replies)]:
    ans = gather(batches, replies)
    if ans:
        answered = [it for it in items if it.id in ans]   # score only what was answered
        rows[cond] = score(answered if len(ans) < len(items) else items, ans, cond)
        log_feedback(fb, rows[cond], MODEL_NAME)          # real, measured outcomes

if rows:
    report = summarize(items, rows)
    print(json.dumps(report, indent=1))
    json.dump({'report': report, 'feedback_log': fb.feedback_log}, open('results_step2.json', 'w'), indent=1)
    print('saved results_step2.json')
else:
    print('No replies pasted yet.')

## Part C. API mode (optional): skip copy-paste
Give `collect_answers` any function that takes a prompt string and returns the reply text. Example shape:
```python
def ask(prompt):
    ...call your provider here and return the text...
answers = collect_answers(items, ask)                      # baseline
answers_g = collect_answers(items, ask, cb.contexts())     # grounded
```

## Part D. The live loop on a NEW claim (verifier overrides a wrong model)

In [ ]:
fb_live = AIFeedbackInterface()
overconfident = lambda prompt: 'TRUE'        # stand-in for a model that agrees with everything
print(answer_with_verification(overconfident, engine, fb_live, 'live-1', '(x+y)**2', 'x**2 + y**2'))
print(answer_with_verification(overconfident, engine, fb_live, 'live-2', 'sin(x)**2 + cos(x)**2', '1'))
print(fb_live.feedback_log[0]['outcome'], fb_live.feedback_log[1]['outcome'])